# Day 1-02：PyTorch入門
**到達目標**：テンソルのshapeを読み、部分を取得・更新し、各行の予測クラスを計算できる。
各例でshapeを予想してから実行し、dimや添字を変えて結果を確かめます。
最後の勾配の節は、MLPの学習へ進む直前に戻ってきても構いません。

## 実行前の準備
> **⚠️ 初回に必ず確認：セル冒頭の`DRIVE_PROJECT`と`LOCAL_PROJECT`を、自分の環境の絶対パスに変更してください。**

このセルを最初に実行してください。Driveのマウント、教材フォルダへの移動、Colab用パッケージの準備を自動で行います。
ローカルではカーネルに **Python (jkj1a)** を選びます。


In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parents[1]
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)


In [ ]:
import torch
from torch import nn

## 1. テンソルを作る
テンソルは数値を並べたもの。`shape` は各軸の長さ、`dtype` は要素の型です（指定しなくても良い）。

In [ ]:
x = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(x)
print("shape:", x.shape, "dtype:", x.dtype, "device:", x.device)

## 2. 部分を取り出す
行・列の順に添字を書きます。整数添字ではその軸が消え、スライスでは残ります。

In [ ]:
print("x:", x)
print("1行目（添字0）:", x[0], x[0].shape)
print("1行だけの行列:", x[0:1], x[0:1].shape)
print("2列目:", x[:, 1])
print("右上の部分:\n", x[:2, 2:])

## 3. 部分に代入する／コピーを作る
`y = x` はコピーではありません（`y`を変更したら`x`も変わる）。独立したテンソルが欲しい場合は `clone()` を使います。
基本的なスライスも元データを共有するので注意します。

In [ ]:
y = x.clone()
y[:2, 1:3] = -1
print("変更したy:\n", y)
print("元のx:\n", x)
part = y[0]
part[0] = 99
print("partの変更はyにも反映:\n", y)

## 4. テンソル全体に関数を適用する
Pythonのfor文を各要素に書かなくても計算できます。ReLUは負の値を0にします。

In [ ]:
a = torch.tensor([-2.0, -0.5, 0.0, 1.0, 3.0])
print("ReLU:", torch.relu(a))
print("二乗:", a.square())
print("正の要素:", a[a > 0])
b = a.clone()
b[b < 0] = 0
print("条件で代入:", b)

## 5. dim：どの軸に沿ってまとめるか
`dim=0` は行の軸、`dim=1` は列の軸をまとめます。
問い：以下のmeanは、それぞれ何個の値を返すでしょう？

In [ ]:
print(x)
print("全体平均:", x.mean())
print("列ごとの平均:", x.mean(dim=0))
print("行ごとの平均:", x.mean(dim=1))
print("行ごとの平均、軸を残す:\n", x.mean(dim=1, keepdim=True))
centered = x - x.mean(dim=0)  # 列ごとの平均を全行から引く（broadcast）
print("中心化:\n", centered)

## 6. argmax：最大値の「場所」から分類を決める
1行が1サンプル、1列が1クラスのスコアです。`argmax` は値そのものではなく添字を返します。
同点なら最初の最大値の添字になります。

In [ ]:
logits = torch.tensor([[0.1, 1.2, -0.3], [2.0, 0.3, 1.1], [-1.0, 0.2, 1.5]])
labels = torch.tensor([1, 0, 0])
predictions = logits.argmax(dim=1)
print("予測クラス:", predictions)
print("行ごとの最大値:", logits.max(dim=1).values)
print("正解したか:", predictions == labels)
print("正解率:", (predictions == labels).float().mean().item())

## 7. 小さなMLPにバッチを渡す
入力 `[サンプル数, 特徴数]` → 出力 `[サンプル数, クラス数]`。
`Linear` は重み付き和とバイアス、`ReLU` は要素ごとの非線形変換です。

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 2))
inputs = torch.tensor([[0.0, 1.0], [1.0, 0.0], [0.5, 0.5]])
outputs = model(inputs)
print(model)
print("input:", inputs.shape, "output:", outputs.shape)
print("predictions:", outputs.argmax(dim=1))

## 8. 自動微分
`backward()` で微分を計算します。勾配は加算されるため、次の計算前に消す必要があります。

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w - 5) ** 2
loss.backward()
print("loss:", loss.item(), "gradient:", w.grad.item())
with torch.no_grad():
    w -= 0.1 * w.grad
w.grad = None
print("更新後のw:", w.item(), "更新後のloss:", ((w-5)**2).item())

## 次は練習
**03-tensor-exercises.ipynb** を開いてください。
まず実行結果の形・値を予想し、自分でコードを書いてから動かします。